# Deep Research with agentcache

This notebook demonstrates how to use the `agentcache` library interactively to perform deep, multi-angle research. 

It uses parallel cache-safe forks to investigate a topic without polluting the parent session's context or breaking the prompt cache.

In [ ]:
!pip install python-dotenv nest-asyncio "git+https://github.com/masterFoad/agentcache.git@main#egg=agentcache[cli]"

In [ ]:
import asyncio
import nest_asyncio
import os
import re
from IPython.display import display, Markdown
from dotenv import load_dotenv

# Required for running asyncio loops inside Jupyter Notebooks
nest_asyncio.apply()
load_dotenv()

from agentcache import AgentSession, ForkPolicy, LiteLLMSDKProvider

### 1. Initialize the Lead Session

We create a lead session with a comprehensive system prompt. This prompt establishes the research methodology and provides enough tokens to activate automatic prompt caching (e.g., Anthropic or OpenAI's 1,024-token minimum).

In [ ]:
SYSTEM_PROMPT = (
    "You are a senior research analyst. You break complex questions into "
    "distinct investigation angles, then produce structured reports with "
    "evidence-backed findings.\n\n"
    "Research methodology:\n"
    "1. Identify 3-4 orthogonal angles that together cover the question\n"
    "2. For each angle, gather key facts, competing theories, and evidence\n"
    "3. Synthesize findings into a coherent narrative with confidence levels\n"
    "4. Flag unresolved questions and areas needing further investigation\n\n"
    "Output style: clear, dense, no filler. Use bullet points for findings. "
    "Cite specific names, dates, and mechanisms rather than vague summaries.\n\n"
    # Padding to ensure we cross the prompt-caching threshold
    + "".join(
        f"Reference principle #{i}: Multi-agent research should use isolated, "
        f"ephemeral forks that share a cacheable prefix. (principle {i})\n"
        for i in range(50)
    )
)

provider = LiteLLMSDKProvider()

# Use gpt-4o-mini or gemini-2.5-flash if available
model = "gpt-4o-mini" if os.getenv("OPENAI_API_KEY") else "gemini/gemini-2.5-flash"

session = AgentSession(
    model=model,
    provider=provider,
    system_prompt=SYSTEM_PROMPT,
)

print(f"Model initialized: {model}")
print(f"System prompt size: ~{len(SYSTEM_PROMPT)//4:,} estimated tokens")

### 2. Plan the Research

The user provides a topic. The lead session breaks it down into distinct investigation angles.

In [ ]:
TOPIC = "What are the physical limits to miniaturization in silicon-based semiconductor manufacturing?"

async def plan_research():
    print("Planning research angles...")
    response = await session.respond(
        f"Break this research question into 3-4 distinct investigation angles. "
        f"For each angle, write one line: the angle name and a one-sentence scope.\n\n"
        f"Question: {TOPIC}\n\n"
        f"Format each angle exactly as: '1. Name: scope'"
    )
    return response.text

plan_text = asyncio.run(plan_research())
display(Markdown(f"### Research Angles\n\n{plan_text}"))

angles = []
for line in plan_text.strip().splitlines():
    match = re.match(r"^\s*\d+[\.\)]\s*(.+)", line.strip())
    if match:
        angles.append(match.group(1).strip())

### 3. Launch Parallel Workers

We use `asyncio.gather` to launch N parallel `session.fork()` calls. 

Each fork inherits the parent session's exact prompt prefix (so it hits the prompt cache) but runs in an isolated context and skips writing its own transcript to the cache (`ForkPolicy.cache_safe_ephemeral()`).

In [ ]:
async def run_worker(angle: str):
    prompt = (
        f"Investigate this specific angle of the research question.\n\n"
        f"Original question: {TOPIC}\n"
        f"Your angle: {angle}\n\n"
        f"Return a structured report with:\n"
        f"- 3-5 key findings (bullet points, specific facts)\n"
        f"- Competing theories or open debates\n"
        f"- Confidence level (high/medium/low) with brief justification\n\n"
        f"Be specific. Cite numbers, mechanisms, and physics constraints."
    )
    return await session.fork(
        prompt=prompt,
        policy=ForkPolicy.cache_safe_ephemeral(),
    )

async def run_all_workers():
    print(f"Launching {len(angles)} parallel workers...")
    tasks = [run_worker(angle) for angle in angles]
    return await asyncio.gather(*tasks)

worker_results = asyncio.run(run_all_workers())
print("All workers complete!")

### 4. Synthesize Findings

We run one final cache-safe fork. It consumes the structured reports from the workers, synthesizing them into a final executive brief.

In [ ]:
async def synthesize():
    worker_reports = "\n\n---\n\n".join(
        f"## Angle {i+1}: {angle}\n\n{result.final_text}"
        for i, (angle, result) in enumerate(zip(angles, worker_results))
    )
    
    synthesis = await session.fork(
        prompt=(
            f"You have received reports from {len(angles)} research workers "
            f"investigating different angles of this question:\n\n"
            f"Question: {TOPIC}\n\n"
            f"Worker reports:\n\n{worker_reports}\n\n"
            f"Synthesize these into a final research brief:\n"
            f"1. Executive summary (3-4 sentences)\n"
            f"2. Key findings across all angles\n"
            f"3. Consensus limits and hard physical boundaries\n"
            f"4. Alternative materials/architectures mentioned"
        ),
        policy=ForkPolicy.cache_safe_ephemeral(),
    )
    return synthesis

final_report = asyncio.run(synthesize())
display(Markdown(final_report.final_text))

### 5. Review Cache Performance and Usage

Because all workers shared the parent's prefix, the later queries (including the synthesis) should show a high cache hit rate, saving tokens and lowering costs.

In [ ]:
status = session.cache_status()
print("=== Session Cache Status ===")
print(status.pretty())
print("\n=== Worker Usage ===")
for i, result in enumerate(worker_results):
    print(f"Worker {i+1}: {result.usage.total_tokens:,} total tokens | {result.usage.cache_read_input_tokens:,} cached tokens")
print(f"\nSynthesis Usage: {final_report.usage.total_tokens:,} total tokens | {final_report.usage.cache_read_input_tokens:,} cached tokens")
print("\nCache Break Diagnosis:")
explanation = session.explain_last_cache_break()
if explanation:
    print(explanation.pretty())
else:
    print("No cache breaks detected.")